# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a line chart because it clearly shows how population growth has changed over time for all six continents. Indexing each continent to 1970 = 100 makes their growth rates comparable even though their starting populations differ.

In [2]:
# Q1
continent_totals = (df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum())
continent_totals['Indexed population'] = (
    continent_totals.groupby('Continent')['Population']
    .transform(lambda s: s / s.iloc[0] * 100)
)

accent = {'Africa': '#0072B2', 'Asia': '#6A3D9A', 'Europe': '#D55E00',
          'North America': '#009E73', 'Oceania': '#CC79A7', 'South America': '#E69F00'}
fig = px.line(continent_totals, x='Year', y='Indexed population', color='Continent',
              color_discrete_map=accent, template='plotly_white')
fig.update_traces(line_width=3)
fig.update_layout(title="Africa has grown fastest since 1970 — reaching over 3.6 times its 1970 population",
                  xaxis_title=None, yaxis_title=None, legend=None,
                  margin=dict(l=55, r=30, t=75, b=55), showlegend=False,
                  yaxis=dict(ticksuffix='', zeroline=False, gridcolor='#E6E6E6'))

latest = continent_totals[continent_totals['Year'] == 2022].sort_values('Indexed population')
for _, row in latest.iterrows():
    fig.add_annotation(x=row['Year'], y=row['Indexed population'], text=row['Continent'],
                       showarrow=False, xanchor='left', xshift=6,
                       font=dict(size=11, color=accent[row['Continent']]))
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I used a sorted horizontal bar chart because it presents the 15 largest absolute population gains in a clear ranking and makes differences between countries easy to compare. I highlighted African countries in blue and used grey for the other continents so that continent membership adds useful context without distracting from the rankings.

In [3]:
# Q2
df['Population added'] = df['2022 Population'] - df['1970 Population']
top15 = df.nlargest(15, 'Population added').sort_values('Population added')
top15['Africa?'] = np.where(top15['Continent'].eq('Africa'), 'Africa', 'Other continents')

fig = px.bar(top15, x='Population added', y='Country', color='Africa?',
             color_discrete_map={'Africa': '#0072B2', 'Other continents': '#A0A0A0'},
             orientation='h', template='plotly_white')
fig.update_traces(texttemplate='%{x:.2s}', textposition='outside', cliponaxis=False)
africa_count = (top15['Africa?'] == 'Africa').sum()
fig.update_layout(title=f"India and China dominate the top gains; {africa_count} of the 15 countries are in Africa",
                  xaxis_title=None, yaxis_title=None, legend=None,
                  margin=dict(l=125, r=55, t=75, b=55),
                  xaxis=dict(gridcolor='#E6E6E6', zeroline=False, tickformat='~s'))
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I selected a bubble scatter plot because it compares land area with 2022 population while also allowing population share to be represented by bubble size. Using logarithmic axes reduces the effect of very large countries and makes the overall pattern easier to examine.

In [4]:
# Q3
palette = {'Africa': '#0072B2', 'Asia': '#D55E00', 'Europe': '#009E73',
           'North America': '#CC79A7', 'Oceania': '#6A3D9A', 'South America': '#E69F00'}
fig = px.scatter(df, x='Area', y='2022 Population', size='World share (%)',
                 color='Continent', hover_name='Country',
                 color_discrete_map=palette, log_x=True, log_y=True,
                 template='plotly_white', size_max=28)
fig.update_layout(title="Area and population have only a weak relationship; China and India are major high-population outliers",
                  xaxis_title=None, yaxis_title=None, legend=None,
                  margin=dict(l=65, r=30, t=75, b=55),
                  xaxis=dict(gridcolor='#E6E6E6'), yaxis=dict(gridcolor='#E6E6E6'))
outliers = df[(df['2022 Population'] > 300_000_000) | ((df['Area'] > 7_000_000) & (df['2022 Population'] > 100_000_000))]
for row in outliers.iterrows():
    r = row[1]
    fig.add_annotation(x=r['Area'], y=r['2022 Population'], text=r['Country'],
                       showarrow=True, arrowhead=2, arrowsize=0.7, ax=20, ay=-25,
                       font=dict(size=10))
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a violin plot because the question asks about variation, not only average density. It displays the distribution, median, and spread for each continent, making it possible to identify skewed distributions and unusually dense countries.

In [5]:
# Q4
order = df.groupby('Continent')['Density'].median().sort_values(ascending=False).index.tolist()
fig = px.violin(df, x='Continent', y='Density', category_orders={'Continent': order},
                box=True, points='outliers', template='plotly_white',
                color_discrete_sequence=['#0072B2'])
fig.update_traces(marker=dict(size=4, color='#4D4D4D', opacity=0.45),
                  line=dict(color='#0072B2'), fillcolor='rgba(0,114,178,0.18)',
                  spanmode='hard')
fig.update_layout(title="Asia is densely concentrated, with Europe and island-heavy regions creating a high-density upper tail",
                  xaxis_title=None, yaxis_title=None, showlegend=False,
                  margin=dict(l=65, r=30, t=75, b=55),
                  yaxis=dict(type='log', gridcolor='#E6E6E6', zeroline=False))
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a choropleth map because it shows percentage population change in a geographic context, with one value for each country. The blue-to-brown diverging scale distinguishes growth from decline while keeping countries with little or no change visually neutral.

In [6]:
# Q5
map_df = df.copy()
map_df['Population change (%)'] = ((map_df['2022 Population'] - map_df['2000 Population']) /
                                   map_df['2000 Population'] * 100)
fig = px.choropleth(map_df, locations='CCA3', locationmode='ISO-3',
                    color='Population change (%)', hover_name='Country',
                    color_continuous_scale=[[0, '#A6611A'], [0.5, '#F0F0F0'], [1, '#0571B0']],
                    range_color=(-30, 150), template='plotly_white')
fig.update_layout(title="Most population growth is concentrated in Africa and Asia; several European and Asian countries declined",
                  margin=dict(l=20, r=20, t=75, b=20),
                  coloraxis_colorbar=dict(title='Change<br>(%)', ticksuffix='%'))
fig.update_geos(showframe=False, showcoastlines=False, showland=True,
                landcolor='#F5F5F5', bgcolor='white')
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I used a treemap because it represents both continents and their countries within a single hierarchy. Area communicates population size, while the limited set of continent colours helps the reader distinguish which countries belong to each region.

In [7]:
# Q6
palette = {'Africa': '#0072B2', 'Asia': '#D55E00', 'Europe': '#009E73',
           'North America': '#CC79A7', 'Oceania': '#6A3D9A', 'South America': '#E69F00'}
fig = px.treemap(df, path=[px.Constant('World'), 'Continent', 'Country'], values='2022 Population',
                 color='Continent', color_discrete_map=palette, template='plotly_white')
fig.update_traces(texttemplate='%{label}<br>%{value:.3s}', textfont_size=11,
                  marker_line_width=1, marker_line_color='white')
fig.update_layout(title="Asia holds 58.7% of the world population; India and China alone account for more than 35%",
                  margin=dict(l=10, r=10, t=75, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a 100% stacked bar chart because it normalizes every year to 100% and makes changes in the continents' relative shares easy to compare. Separate bars retain the chronological order, while the stacking directly emphasizes the changing population composition.

In [8]:
# Q7
share = (df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum())
world = share.groupby('Year')['Population'].transform('sum')
share['Share (%)'] = share['Population'] / world * 100
share['Group'] = share['Continent'].where(share['Continent'].isin(['Africa', 'Asia', 'Europe']),
                                          'Rest of the world')
share = share.groupby(['Year', 'Group'], as_index=False)['Share (%)'].sum()

palette = {'Africa': '#0072B2', 'Asia': '#D55E00', 'Europe': '#009E73',
           'Rest of the world': '#BDBDBD'}
fig = px.bar(share, x='Year', y='Share (%)', color='Group', barmode='relative',
             color_discrete_map=palette, template='plotly_white')
fig.update_layout(title="Africa's world share rose; Europe's fell while Asia remains the world's largest share",
                  xaxis_title=None, yaxis_title=None, legend_title=None,
                  margin=dict(l=55, r=25, t=75, b=55),
                  yaxis=dict(range=[0, 100], ticksuffix='%', gridcolor='#E6E6E6'))
fig.update_traces(width=1.8)
fig.show()